# Пара 6. Стилиометрия mini-Delta

Сравните **два фрагмента** (две главы диссертации / два рассказа / «ваш автор» vs контроль).  
Результат — **гипотеза близости**, не доказательство авторства.


## Старт в Colab
1. **Файл → Сохранить копию на Диске** (чтобы правки не пропали).
2. Справа сверху — **Подключить**.
3. Ячейки сверху вниз: Play или Shift+Enter.
Базовый Colab **бесплатный**. Правило занятия: **вопрос → процедура → картинка → цитата → ограничение**.


In [ ]:
import re, requests, math
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

def tokenize(t):
    return re.findall(r"[А-Яа-яЁё]+", t.lower())

def load(url):
    r = requests.get(url, timeout=60); r.encoding = "utf-8"; return r.text

# DEMO: три рассказа Чехова. Для диссертации — три своих .txt
CHEKHOV = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/chekhov-rasskazy/txt/"
docs = {
    "Дама с собачкой": load(CHEKHOV + "dama-s-sobachkoj.txt"),
    "Ионыч": load(CHEKHOV + "ionyich.txt"),
    "Палата №6": load(CHEKHOV + "palata-6.txt"),
}

# MFW по объединённому корпусу
all_toks = []
freqs = {}
for name, text in docs.items():
    toks = tokenize(text)
    freqs[name] = Counter(toks)
    all_toks.extend(toks)
mfw = [w for w, _ in Counter(all_toks).most_common(50)]

# относительные частоты
rel = {name: {w: freqs[name][w] / max(sum(freqs[name].values()), 1) for w in mfw} for name in docs}

# среднее и sigma по корпусу
import statistics
means = {w: statistics.mean(rel[n][w] for n in docs) for w in mfw}
sigmas = {w: statistics.pstdev(rel[n][w] for n in docs) or 1e-9 for w in mfw}

def zvec(name):
    return {w: (rel[name][w] - means[w]) / sigmas[w] for w in mfw}

Z = {n: zvec(n) for n in docs}

def delta(a, b):
    return sum(abs(Z[a][w] - Z[b][w]) for w in mfw) / len(mfw)

names = list(docs)
mat = pd.DataFrame([[delta(a, b) for b in names] for a in names], index=names, columns=names)
mat.round(3)


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(mat.values, cmap="viridis_r")
plt.xticks(range(len(names)), names, rotation=30, ha="right")
plt.yticks(range(len(names)), names)
plt.colorbar(label="mini-Delta")
plt.title("Чем меньше, тем ближе профиль MFW")
plt.tight_layout(); plt.show()


## Сдать: матрица + 5 предложений: жанр/тема могут объяснить близость лучше, чем «автор».
